In [2]:
%pip install python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 30.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [python-pptx] [python-pptx]
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN

def create_ak_sk_ppt():
    # 1. 初始化演示文稿
    prs = Presentation()

    # 设置幻灯片布局 (0: 标题页, 1: 标题+内容)
    title_slide_layout = prs.slide_layouts[0]
    content_slide_layout = prs.slide_layouts[1]

    # --- 第1页：标题页 ---
    slide = prs.slides.add_slide(title_slide_layout)
    title = slide.shapes.title
    subtitle = slide.placeholders[1]
    title.text = "AK/SK 认证机制深度解析"
    subtitle.text = "基于对称加密的请求合法性验证方案\n生成时间：2026-03-07"

    # --- 第2页：什么是 AK/SK？ ---
    slide = prs.slides.add_slide(content_slide_layout)
    slide.shapes.title.text = "1. 核心组件定义"
    content = slide.placeholders[1]
    tf = content.text_frame
    tf.text = "AK/SK 是身份验证的“双子星”："
    
    p = tf.add_paragraph()
    p.text = "• Access Key (AK): 身份标识，相当于用户名（公开）。"
    p.level = 1
    
    p = tf.add_paragraph()
    p.text = "• Secret Key (SK): 访问密钥，相当于密码（严禁泄露，不参与传输）。"
    p.level = 1
    
    p = tf.add_paragraph()
    p.text = "• 核心逻辑: 双方持有相同的 SK，通过对请求内容计算哈希值（签名）来建立信任。"
    p.level = 0

    # --- 第3页：认证流程图解 ---
    slide = prs.slides.add_slide(content_slide_layout)
    slide.shapes.title.text = "2. 四步签名流程 (SigV4 逻辑)"
    rows, cols = 5, 2
    left, top, width, height = Inches(0.5), Inches(1.5), Inches(9), Inches(3)
    table = slide.shapes.add_table(rows, cols, left, top, width, height).table
    
    # 设置表格标题
    table.cell(0, 0).text = "步骤"
    table.cell(0, 1).text = "操作内容"
    
    data = [
        ("1. 规范化", "对 Method, URI, Query, Payload 进行排序格式化"),
        ("2. 待签名串", "将规范化结果与时间戳、凭证范围结合做 Hash"),
        ("3. 派生密钥", "使用 SK 逐层计算衍生 Key (Date/Region/Service)"),
        ("4. 生成签名", "使用派生 Key 对待签名串进行最终 HMAC-SHA256")
    ]
    
    for i, (step, desc) in enumerate(data, start=1):
        table.cell(i, 0).text = step
        table.cell(i, 1).text = desc

    # --- 第4页：安全性分析 ---
    slide = prs.slides.add_slide(content_slide_layout)
    slide.shapes.title.text = "3. 安全性保障维度"
    content = slide.placeholders[1]
    tf = content.text_frame
    
    points = [
        ("防篡改", "请求内容的任何变动都会导致本地计算的签名不一致。"),
        ("防重放", "强制包含请求时间戳，过期请求（如 >5min）直接丢弃。"),
        ("防泄露", "Secret Key 永远不在网络中传输，只传哈希结果。"),
        ("防反推", "基于 HMAC-SHA256 算法，无法通过签名反向推导 SK。")
    ]
    
    for title_pt, desc_pt in points:
        p = tf.add_paragraph()
        p.text = f"• {title_pt}: {desc_pt}"
        p.level = 0

    # --- 第5页：代码实现核心 ---
    slide = prs.slides.add_slide(content_slide_layout)
    slide.shapes.title.text = "4. Python 实现参考"
    content = slide.placeholders[1]
    content.text = (
        "import hmac, hashlib\n"
        "# 核心签名公式\n"
        "signature = hmac.new(\n"
        "    signing_key, \n"
        "    string_to_sign.encode('utf-8'), \n"
        "    hashlib.sha256\n"
        ").hexdigest()"
    )
    # 调整字体为等宽
    for paragraph in content.text_frame.paragraphs:
        for run in paragraph.runs:
            run.font.name = 'Courier New'
            run.font.size = Pt(18)

    # --- 保存文件 ---
    file_name = "AK_SK_Auth_Analysis.pptx"
    prs.save(file_name)
    print(f"成功生成 PPT: {file_name}")

if __name__ == "__main__":
    create_ak_sk_ppt()

成功生成 PPT: AK_SK_Auth_Analysis.pptx
